# Preprocess for the reference cell type

In [1]:
import pandas as pd
import scanpy as sc

In [2]:
celltypist = sc.read_h5ad('../celltypist/raw_data/global/CountAdded_PIP_global_object_for_cellxgene.h5ad')
celltypist

AnnData object with n_obs × n_vars = 329762 × 36601
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range'
    uns: 'Age_range_colors', 'Sex_colors'
    obsm: 'X_umap'
    layers: 'counts'

In [3]:
cellt = pd.DataFrame(celltypist.obs['Predicted_labels_CellTypist'].value_counts())
cellt

,count
Predicted_labels_CellTypist,
Cytotoxic T cells,36883
Memory B cells,31194
Helper T cells,25761
Macrophages,24071
CD16+ NK cells,21495
...,...
Transitional DC,41
Cycling monocytes,31
Myelocytes,25


In [4]:
cell = pd.read_csv('./origin/cellType_level.csv')
cell = cell.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
cell

/tmp/ipykernel_1370808/2452545847.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cell = cell.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)


,level_1,level_2,level_3
0,HSC/MPP,HSC/MPP,HSC/MPP
1,HSC/MPP,HSC/MPP,CMP
2,HSC/MPP,HSC/MPP,GMP
3,HSC/MPP,HSC/MPP,ELP
4,HSC/MPP,HSC/MPP,MEMP
...,...,...,...
94,others,Cycling cells,Cycling gamma-delta T cells
95,others,Cycling cells,Cycling monocytes
96,others,Cycling cells,Cycling NK cells
97,others,Cycling cells,Cycling T cells


In [5]:
ss = pd.DataFrame(set(cellt.index.tolist()) & set(cell['level_3'].tolist()))
ss.columns = ['celltype']

In [6]:
ss

,celltype
0,Classical monocytes
1,MAIT cells
2,GMP
3,Mono-mac
4,Tem/Effector helper T cells PD1+
...,...
80,Neutrophils
81,Late erythroid
82,Fibroblasts
83,B cells


In [7]:
cells = pd.merge(cell, ss, left_on='level_3', right_on='celltype')
cells = cells[['level_1','level_2','level_3']]
cells

,level_1,level_2,level_3
0,HSC/MPP,HSC/MPP,HSC/MPP
1,HSC/MPP,HSC/MPP,CMP
2,HSC/MPP,HSC/MPP,GMP
3,HSC/MPP,HSC/MPP,ELP
4,HSC/MPP,HSC/MPP,MEMP
...,...,...,...
80,others,Cycling cells,Cycling DCs
81,others,Cycling cells,Cycling gamma-delta T cells
82,others,Cycling cells,Cycling monocytes
83,others,Cycling cells,Cycling NK cells


In [8]:
cells['level_1'].value_counts()

level_1
Lymphoid    37
Myeloid     31
others       9
HSC/MPP      8
Name: count, dtype: int64

In [11]:
oth = pd.DataFrame(set(cellt.index.tolist()) - set(ss['celltype'].tolist()))
oth.columns = ['level_3']
oth['level_2'] = ['B cells','B cells','T cells','T cells','T cells','T cells']
oth['level_1'] = ['Lymphoid']*6

In [12]:
oth = oth[['level_1','level_2','level_3']]
oth

,level_1,level_2,level_3
0,Lymphoid,B cells,Immature B cells
1,Lymphoid,B cells,Pre-B cells
2,Lymphoid,T cells,T cells
3,Lymphoid,T cells,Cytotoxic T cells
4,Lymphoid,T cells,Tem/Effector cytotoxic T cells
5,Lymphoid,T cells,Helper T cells


In [13]:
ot = pd.DataFrame(set(cell['level_3'].tolist()) - set(cellt.index.tolist()))
ot

,0
0,Age-associated B cells
1,Tem/Temra cytotoxic T cells
2,Trm cytotoxic T cells
3,CRTAM+ gamma-delta T cells
4,Alveolar macrophages
5,Proliferative germinal center B cells
6,Erythrophagocytic macrophages
7,notAvailable
8,Large pre-B cells
9,Small pre-B cells


In [14]:
cell_a = pd.concat([cell, oth])
cell_a

,level_1,level_2,level_3
0,HSC/MPP,HSC/MPP,HSC/MPP
1,HSC/MPP,HSC/MPP,CMP
2,HSC/MPP,HSC/MPP,GMP
3,HSC/MPP,HSC/MPP,ELP
4,HSC/MPP,HSC/MPP,MEMP
...,...,...,...
1,Lymphoid,B cells,Pre-B cells
2,Lymphoid,T cells,T cells
3,Lymphoid,T cells,Cytotoxic T cells
4,Lymphoid,T cells,Tem/Effector cytotoxic T cells


In [15]:
cell_a.to_csv('./origin/sc_105.csv', index=None)

# Collate the results of eight algorithms

In [ ]:
import pandas as pd
import scanpy as sc
import os

## singler

In [10]:
part = pd.read_csv('./singler/predicted_celltype_2.csv', index_col=0)
part

,V1,V2
1,Cheong_Cell_2023_1-ATGACTCAGGCATTAC-jcov127,CD16+ NK cells
2,Cheong_Cell_2023_1-ATGACTCAGGTGAAGC-jcov127,Granulocytes
3,Cheong_Cell_2023_1-ATGAGCCGTACTGAAT-jcov127,Naive B cells
4,Cheong_Cell_2023_1-ATGAGCCGTATTCGCT-jcov127,CD16+ NK cells
5,Cheong_Cell_2023_1-ATGAGCCGTCTTTATG-jcov127,HSC/MPP
...,...,...
99996,Cheong_Cell_2023_1-CAAGGGAGTGACCTGG-lgtd8m,HSC/MPP
99997,Cheong_Cell_2023_1-CAAGGGAGTGTCCAAA-lgtd8m,Classical monocytes
99998,Cheong_Cell_2023_1-CAAGGGAGTGTCCAGG-lgtd8m,Classical monocytes
99999,Cheong_Cell_2023_1-CAAGGTAAGACTCGCA-lgtd8m,Classical monocytes


## scpred

In [2]:
folder_path = "./scpred/"
dataframes = []

In [3]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path,index_col=0)
        dataframes.append(df)

In [4]:
merged_df = pd.concat(dataframes, axis=0, ignore_index=True)

In [6]:
merged_df.columns = ['cellname', 'celltype']

In [8]:
merged_df.index = merged_df['cellname'].tolist()

In [9]:
merged_df

,cellname,celltype
Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Early erythroid
Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Pro-B cells
Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Granulocytes
Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Granulocytes
Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Mast cells
...,...,...
Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,unassigned
Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Granulocytes
Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Late erythroid
Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Promyelocytes


In [10]:
merged_df['celltype'].value_counts()

celltype
unassigned                   1513512
Mast cells                    109288
Regulatory T cells             90955
Granulocytes                   81649
CD16+ NK cells                 62692
                              ...   
Follicular helper T cells          3
Transitional B cells               3
ILC precursor                      2
Treg(diff)                         2
CD8a/a                             1
Name: count, Length: 87, dtype: int64

In [11]:
merged_df.to_csv('scpred_all.csv')

In [13]:
data = pd.read_csv('/pool1/liumengmeng/project_blood_scATAC/cell_annotation/scPred/celltype_summary/combined_data.csv')
data

,Unnamed: 0,X,scpred_no_rejection
0,predicted_celltype_1.csv.1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Regulatory T cells
1,predicted_celltype_1.csv.2,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Regulatory T cells
2,predicted_celltype_1.csv.3,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Helper T cells
3,predicted_celltype_1.csv.4,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Helper T cells
4,predicted_celltype_1.csv.5,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Helper T cells
...,...,...,...
2462623,predicted_celltype_9.csv.99996,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Tem/Effector helper T cells
2462624,predicted_celltype_9.csv.99997,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,HSC/MPP
2462625,predicted_celltype_9.csv.99998,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Transitional B cells
2462626,predicted_celltype_9.csv.99999,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Plasma cells


In [2]:
data = pd.read_csv('/pool1/liumengmeng/project_blood_scATAC/cell_annotation/scPred/cell_summary_glm/combined_data.csv')
data

,Unnamed: 0,X,orig.ident,nCount_RNA,nFeature_RNA,scpred_DC,scpred_T.cells,scpred_Memory.B.cells,scpred_Macrophages,scpred_Tcm.Naive.helper.T.cells,...,scpred_Neutrophils,scpred_Migratory.DCs,scpred_Pre_minusB.cells,scpred_ILC.precursor,scpred_Transitional.DC,scpred_Cycling.DCs,scpred_Myelocytes,scpred_max,scpred_prediction,scpred_no_rejection
0,predicted_input_1.csv.1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen,2407,1395,0.000852,0.018523,0.009592,0.000879,0.070769,...,2.525143e-05,2.839221e-04,2.220446e-16,0.000144,3.031552e-06,2.220446e-16,2.220446e-16,0.151229,Regulatory T cells,Regulatory T cells
1,predicted_input_1.csv.2,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen,2126,1270,0.002302,0.006722,0.001176,0.000218,0.020698,...,5.303209e-05,2.909880e-05,2.220446e-16,0.000192,1.219710e-05,2.220446e-16,2.220446e-16,0.506100,Regulatory T cells,Regulatory T cells
2,predicted_input_1.csv.3,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen,1996,1153,0.000691,0.037259,0.003908,0.003798,0.082188,...,1.137611e-03,3.953842e-05,2.220446e-16,0.000831,7.157190e-05,2.220446e-16,2.220446e-16,0.181316,Helper T cells,Helper T cells
3,predicted_input_1.csv.4,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen,1949,1113,0.004119,0.031829,0.028404,0.002961,0.152614,...,2.687572e-03,3.290664e-03,2.220446e-16,0.000843,6.402428e-04,2.220446e-16,2.220446e-16,0.349380,Helper T cells,Helper T cells
4,predicted_input_1.csv.5,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen,3041,1674,0.002444,0.012200,0.005100,0.000680,0.077862,...,2.575772e-04,2.663137e-04,2.220446e-16,0.000318,1.309063e-04,2.220446e-16,2.220446e-16,0.083322,unassigned,Helper T cells
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462623,predicted_input_9.csv.99996,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Weng,1860,1198,0.000104,0.011047,0.000435,0.000565,0.061843,...,2.164716e-04,3.225810e-05,2.220446e-16,0.000204,1.253959e-05,2.220446e-16,2.220446e-16,0.077772,unassigned,Tem/Effector helper T cells
2462624,predicted_input_9.csv.99997,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Weng,4615,2139,0.002472,0.000153,0.000001,0.000007,0.000006,...,1.641670e-09,2.288215e-01,2.220446e-16,0.000182,1.662527e-02,2.220446e-16,2.220446e-16,0.999994,HSC/MPP,HSC/MPP
2462625,predicted_input_9.csv.99998,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Weng,1630,1126,0.000100,0.005021,0.057753,0.000122,0.000954,...,1.025863e-04,6.273278e-09,2.220446e-16,0.000270,2.220446e-16,2.220446e-16,2.220446e-16,0.084303,unassigned,Transitional B cells
2462626,predicted_input_9.csv.99999,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Weng,2756,1726,0.000920,0.000265,0.000009,0.001541,0.000229,...,2.312226e-07,2.484024e-11,2.220446e-16,0.000001,2.641035e-07,2.220446e-16,2.220446e-16,0.012679,unassigned,Plasma cells


In [7]:
dat = data[['X','scpred_no_rejection']]
dat

,X,scpred_no_rejection
0,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Regulatory T cells
1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Regulatory T cells
2,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Helper T cells
3,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Helper T cells
4,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Helper T cells
...,...,...
2462623,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Tem/Effector helper T cells
2462624,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,HSC/MPP
2462625,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Transitional B cells
2462626,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Plasma cells


In [8]:
dat.columns = ['cellname', 'celltype']

In [10]:
dat.index = dat['cellname'].tolist()
dat

,cellname,celltype
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Helper T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Helper T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Helper T cells
...,...,...
Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Tem/Effector helper T cells
Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,HSC/MPP
Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Transitional B cells
Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Plasma cells


In [11]:
dat.to_csv('scpred_all.csv')

## ACTINN

In [3]:
folder_path = "actinn/"
subfolders = [f.path for f in os.scandir(folder_path) if f.is_dir()][0:25]
print(subfolders)

['actinn/part_1', 'actinn/part_2', 'actinn/part_3', 'actinn/part_4', 'actinn/part_5', 'actinn/part_6', 'actinn/part_7', 'actinn/part_8', 'actinn/part_9', 'actinn/part_10', 'actinn/part_11', 'actinn/part_12', 'actinn/part_13', 'actinn/part_14', 'actinn/part_15', 'actinn/part_16', 'actinn/part_17', 'actinn/part_18', 'actinn/part_19', 'actinn/part_20', 'actinn/part_21', 'actinn/part_22', 'actinn/part_23', 'actinn/part_24', 'actinn/part_25']


In [4]:
i = 0
cell = pd.read_csv(subfolders[i]+'/predicted_label.txt', sep='\t')
cell.index = cell['cellname'].tolist()
pred = pd.read_csv(subfolders[i]+'/predicted_probabilities.txt', sep='\t', index_col=0).T
cell_pred = pd.concat([cell,pred], axis=1)
cell_pred

,cellname,celltype,Hofbauer cells,ILC3,Cycling gamma-delta T cells,B cells,DC,Follicular helper T cells,Type 17 helper T cells,Monocyte precursor,...,Tcm/Naive cytotoxic T cells,Pre-B cells,T(agonist),Macrophages,MNP,ETP,NKT cells,Mono-mac,ELP,DC2
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Regulatory T cells,0.001989,0.036182,0.000410,0.000030,0.000889,6.116328e-04,2.453847e-04,0.001251,...,0.000116,0.000557,0.003310,1.796931e-06,0.001079,0.006590,0.009272,0.000095,0.000183,0.000210
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg(diff),0.000398,0.034033,0.000040,0.000016,0.000528,8.011810e-05,1.179520e-04,0.000312,...,0.000262,0.000452,0.004143,1.051052e-06,0.000205,0.000995,0.032553,0.000032,0.000204,0.000357
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,T cells,0.001481,0.000358,0.000126,0.000018,0.002222,3.951085e-06,7.243298e-06,0.000223,...,0.000358,0.000072,0.000128,1.197694e-07,0.000023,0.000166,0.000102,0.000009,0.000145,0.000166
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,ILC3,0.012480,0.469688,0.000351,0.011805,0.007444,9.962440e-06,6.060800e-04,0.001941,...,0.000016,0.000955,0.001987,6.983764e-07,0.000026,0.000122,0.001447,0.001539,0.002068,0.000369
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Tcm/Naive helper T cells,0.000743,0.148750,0.000104,0.000037,0.002329,2.954562e-04,6.853649e-04,0.000771,...,0.000088,0.000341,0.008393,2.334269e-07,0.000064,0.000524,0.001063,0.000049,0.000104,0.000579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127,Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127,NK cells,0.001400,0.070954,0.000043,0.000290,0.000026,2.075198e-04,5.273911e-04,0.000609,...,0.000002,0.000311,0.017307,3.535429e-05,0.000310,0.000619,0.183910,0.000051,0.000527,0.000327
Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127,Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127,Tem/Effector helper T cells,0.017388,0.116218,0.003666,0.002192,0.000066,1.731659e-01,1.049046e-03,0.003495,...,0.000030,0.001407,0.002579,8.571847e-05,0.000486,0.001709,0.000603,0.000159,0.002121,0.002171
Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127,Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127,Tcm/Naive helper T cells,0.002775,0.016901,0.002725,0.000673,0.017180,2.780347e-04,1.186716e-03,0.001302,...,0.115643,0.002291,0.011263,2.382475e-04,0.001902,0.003786,0.017318,0.001228,0.012592,0.005422
Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127,Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127,Tem/Effector cytotoxic T cells,0.016663,0.033558,0.003879,0.022880,0.000142,7.002957e-03,9.586977e-07,0.002486,...,0.000016,0.000395,0.025237,1.272264e-04,0.000975,0.010661,0.011081,0.000154,0.003775,0.000274


In [6]:
cell_pred[['cellname','celltype']].to_csv('actinn_part1.csv')

In [7]:
for i in range(1,25):
    print(i)
    cell = pd.read_csv(subfolders[i]+'/predicted_label.txt', sep='\t')
    cell.index = cell['cellname'].tolist()
    pred = pd.read_csv(subfolders[i]+'/predicted_probabilities.txt', sep='\t', index_col=0).T
    cell_pre = pd.concat([cell,pred], axis=1)
    cell_pred = pd.concat([cell_pred,cell_pre], axis=0)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24


In [8]:
cell_pred

,cellname,celltype,Hofbauer cells,ILC3,Cycling gamma-delta T cells,B cells,DC,Follicular helper T cells,Type 17 helper T cells,Monocyte precursor,...,Tcm/Naive cytotoxic T cells,Pre-B cells,T(agonist),Macrophages,MNP,ETP,NKT cells,Mono-mac,ELP,DC2
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Regulatory T cells,0.001989,0.036182,0.000410,0.000030,0.000889,6.116328e-04,0.000245,0.001251,...,1.159604e-04,5.572096e-04,0.003310,1.796931e-06,1.078609e-03,0.006590,0.009272,9.529314e-05,0.000183,0.000210
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg(diff),0.000398,0.034033,0.000040,0.000016,0.000528,8.011810e-05,0.000118,0.000312,...,2.621082e-04,4.515319e-04,0.004143,1.051052e-06,2.050176e-04,0.000995,0.032553,3.209589e-05,0.000204,0.000357
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,T cells,0.001481,0.000358,0.000126,0.000018,0.002222,3.951085e-06,0.000007,0.000223,...,3.578412e-04,7.166035e-05,0.000128,1.197694e-07,2.281222e-05,0.000166,0.000102,8.580065e-06,0.000145,0.000166
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,ILC3,0.012480,0.469688,0.000351,0.011805,0.007444,9.962440e-06,0.000606,0.001941,...,1.597248e-05,9.553372e-04,0.001987,6.983764e-07,2.646769e-05,0.000122,0.001447,1.538666e-03,0.002068,0.000369
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Tcm/Naive helper T cells,0.000743,0.148750,0.000104,0.000037,0.002329,2.954562e-04,0.000685,0.000771,...,8.786367e-05,3.405061e-04,0.008393,2.334269e-07,6.448736e-05,0.000524,0.001063,4.870436e-05,0.000104,0.000579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Naive B cells,0.000442,0.004714,0.000219,0.005828,0.001587,4.907626e-08,0.000111,0.000184,...,8.436597e-07,4.151164e-06,0.000168,3.167791e-06,3.019319e-07,0.000031,0.000039,9.836224e-07,0.001053,0.000004
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Transitional B cells,0.016038,0.082289,0.102189,0.001119,0.100390,6.797395e-08,0.000198,0.000169,...,2.736300e-04,1.484632e-06,0.000293,4.858899e-06,1.276056e-04,0.001648,0.074475,3.383468e-04,0.014351,0.008405
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,0.000005,0.000005,0.000009,0.004323,0.000184,4.129443e-09,0.000025,0.000002,...,4.365376e-06,1.888888e-07,0.000013,1.082049e-08,4.351355e-07,0.000025,0.000002,5.448122e-07,0.000011,0.000042
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,NK cells,0.002043,0.014295,0.000744,0.001508,0.002600,1.319137e-05,0.000016,0.000872,...,4.210057e-02,1.691473e-05,0.096445,8.662293e-04,1.989421e-03,0.003980,0.005356,1.587193e-03,0.001141,0.000534


In [9]:
cell_pred['celltype'].value_counts()

celltype
Helper T cells                    297354
Tcm/Naive helper T cells          224954
NK cells                          188827
Tem/Effector cytotoxic T cells    166406
Memory B cells                    166082
                                   ...  
Transitional NK                       14
ILC2                                   7
Erythrocytes                           4
Cycling monocytes                      3
pDC precursor                          2
Name: count, Length: 87, dtype: int64

In [10]:
cell_pred[['cellname','celltype']].to_csv('actinn/actinn_all.csv')

## scmap

In [11]:
# part_1
# 2000 genes, 12.3% NA
# 1000 genes, 12.1% NA
# 500 genes, 9.9%

In [12]:
folder_path = "./scmap/"
dataframes = []

In [1]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path)
        dataframes.append(df)

NameError: name 'os' is not defined

In [14]:
merged_df = pd.concat(dataframes, axis=0, ignore_index=True)

In [15]:
merged_df.columns = ['cellname', 'celltype', 'similarity']
merged_df.index = merged_df['cellname'].tolist()

In [16]:
merged_df

,cellname,celltype,similarity
Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Myelocytes,0.568976
Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Immature B cells,0.771521
Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Early erythroid,0.883716
Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,HSC/MPP,0.875623
Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Germinal center B cells,0.825343
...,...,...,...
Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Tcm/Naive helper T cells,0.752742
Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,HSC/MPP,0.886624
Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Monocytes,0.532146
Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Mid erythroid,0.731983


In [15]:
pd.DataFrame(merged_df['celltype'].value_counts())

,count
celltype,
Neutrophils,1068043
Tcm/Naive helper T cells,236598
unassigned,222356
Early lymphoid/T lymphoid,148273
CD16+ NK cells,76958
...,...
Pro-B cells,20
Endothelial cells,15
Macrophages,7


In [16]:
pd.DataFrame(merged_df['similarity'].isna())['similarity'].value_counts()

similarity
False    2246392
True      216236
Name: count, dtype: int64

In [17]:
merged_df[['cellname', 'celltype']].to_csv('scmap_all.csv')

In [20]:
part_1 = pd.read_csv('../actinn/query/part_1/barcodes.tsv', header=None)
part_1

,0
0,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1
1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1
2,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1
3,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1
4,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1
...,...
99995,Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127
99996,Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127
99997,Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127
99998,Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127


In [23]:
merged_df_part1 = merged_df.loc[part_1[0].tolist(),:]
merged_df_part1

,cellname,celltype,similarity
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,unassigned,NaN
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,unassigned,NaN
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,0.534115
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,0.562835
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Type 17 helper T cells,0.494953
...,...,...,...
Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127,Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127,Fibroblasts,0.832502
Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127,Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127,Neutrophils,0.703722
Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127,Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127,Myelocytes,0.755531
Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127,Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127,unassigned,NaN


In [24]:
merged_df_part1[['cellname', 'celltype']].to_csv('scmap_part1.csv')

## scarch

In [2]:
cell = pd.read_csv('scarch/part_1/preds.csv',index_col=0)
id = pd.read_csv('actinn/part_1/barcodes.tsv', header=None)
cell.index = id[0].tolist()
cell.columns = ['celltype']
cell['cellname'] = id[0].tolist()
cell

,celltype,cellname
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1
...,...,...
Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127,ILC3,Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127
Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127,Treg(diff),Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127
Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127,Myelocytes,Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127
Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127,Hofbauer cells,Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127


In [3]:
cell['celltype'].value_counts()

celltype
Myelocytes                   18987
Monocytes                    14758
Neutrophils                   7217
Treg(diff)                    6095
Early lymphoid/T lymphoid     5745
                             ...  
Mid erythroid                    1
Cycling B cells                  1
Type 1 helper T cells            1
DC3                              1
Cycling DCs                      1
Name: count, Length: 82, dtype: int64

In [4]:
cell[['cellname', 'celltype']].to_csv('scarch_part1.csv')

In [5]:
prefix = "part_"

In [6]:
for i in range(2,26):
    print(i)
    folder_name = f"{prefix}{i}"
    cel = pd.read_csv('scarch/'+folder_name+'/preds.csv',index_col=0)
    id = pd.read_csv('actinn/'+folder_name+'/barcodes.tsv', header=None)
    cel.index = id[0].tolist()
    cel.columns = ['celltype']
    cel['cellname'] = id[0].tolist()
    cell = pd.concat([cell,cel], axis=0)

2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25


In [7]:
cell

,celltype,cellname
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1
...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Weng_Nature_2024_7-TTTGTTGGTCATTACC-1
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1


In [8]:
cell[['cellname', 'celltype']].to_csv('scarch_all.csv')

## chetah unassigned for the most

In [3]:
data = pd.read_csv('chetah/predicted_celltype_1.csv',index_col=0)
data

,celltypes_input,celltypes
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg(diff)
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Early lymphoid/T lymphoid
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Treg(diff)
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Treg(diff)
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Treg(diff)
...,...,...
Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127,Cheong_Cell_2023_1-ATGACGAAGCGCAATT-jcov127,Node15
Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127,Cheong_Cell_2023_1-ATGACGAAGGAACGGT-jcov127,Unassigned
Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127,Cheong_Cell_2023_1-ATGACGAAGGTTTGAC-jcov127,Megakaryocytes/platelets
Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127,Cheong_Cell_2023_1-ATGACGAAGTAATCCA-jcov127,Neutrophils


In [31]:
data.columns = ['cellname', 'celltype']

In [33]:
data[['cellname', 'celltype']].to_csv('chetah_part1.csv')

In [14]:
folder_path = "./chetah/"
dataframes = []

In [15]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path,index_col=0)
        dataframes.append(df)

In [16]:
merged_df = pd.concat(dataframes, axis=0, ignore_index=True)

In [17]:
merged_df

,celltypes_input,celltypes
0,Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Unassigned
1,Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Immature B cells
2,Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Mid erythroid
3,Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Mid erythroid
4,Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Mid erythroid
...,...,...
2462623,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Megakaryocytes/platelets
2462624,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Node10
2462625,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Unassigned
2462626,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Immature B cells


In [18]:
merged_df.columns = ['cellname', 'celltype']
merged_df.index = merged_df['cellname'].tolist()

In [20]:
merged_df.to_csv('chetah_all.csv')

In [14]:
data = pd.read_csv('./chetah_all.csv')
data

,Unnamed: 0,cellname,celltype
0,Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Unassigned
1,Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Immature B cells
2,Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Mid erythroid
3,Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Mid erythroid
4,Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Mid erythroid
...,...,...,...
2462623,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Megakaryocytes/platelets
2462624,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Node10
2462625,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Unassigned
2462626,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Immature B cells


In [15]:
data['celltype'].value_counts()

celltype
Unassigned                  1204666
Megakaryocytes/platelets     334785
Immature B cells             187507
Treg(diff)                   133282
Node10                        96011
                             ...   
Node16                            7
Memory B cells                    6
Kupffer cells                     5
Node32                            2
Node21                            2
Name: count, Length: 170, dtype: int64

## singcellnet

In [3]:
data = pd.read_csv('./singcellnet/combined.csv', index_col=0)
data

,Cell,CellType
1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Tcm/Naive helper T cells
2,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Regulatory T cells
3,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Tcm/Naive helper T cells
4,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Helper T cells
5,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Tcm/Naive helper T cells
...,...,...
2463824,Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Naive B cells
2463825,Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,CD16+ NK cells
2463826,Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells
2463827,Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Cytotoxic T cells


In [4]:
data.index = data['Cell'].tolist()
data

,Cell,CellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Tcm/Naive helper T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Tcm/Naive helper T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Helper T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Tcm/Naive helper T cells
...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Naive B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,CD16+ NK cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Cytotoxic T cells


In [5]:
others = pd.read_csv('./all_method4.csv', index_col=0)
others

,scarch,actinn,celltypist,scmap,mv_3
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Regulatory T cells,Regulatory T cells,unassigned,notAvailable
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Treg(diff),Tem/Effector helper T cells,unassigned,notAvailable
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,T cells,CD16- NK cells,Neutrophils,notAvailable
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,ILC3,Tem/Trm cytotoxic T cells,Neutrophils,notAvailable
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Tcm/Naive helper T cells,Tcm/Naive helper T cells,Type 17 helper T cells,notAvailable
...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Naive B cells,Small pre-B cells,Naive B cells,notAvailable
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Transitional B cells,Small pre-B cells,NK cells,notAvailable
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,NK cells,Tem/Temra cytotoxic T cells,NK cells,notAvailable


In [6]:
len(set(data.index.tolist()) & set(others.index.tolist()))

2462628

In [8]:
data.to_csv('singcellnet_all.csv')

## test

In [3]:
rna = sc.read_h5ad('../6_integrate/processed/adata_rna.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [43]:
actinn = pd.read_csv('actinn_all.csv', index_col=0)
actinn  

,cellname,celltype,Hofbauer cells,ILC3,Cycling gamma-delta T cells,B cells,DC,Follicular helper T cells,Type 17 helper T cells,Monocyte precursor,...,Tcm/Naive cytotoxic T cells,Pre-B cells,T(agonist),Macrophages,MNP,ETP,NKT cells,Mono-mac,ELP,DC2
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Regulatory T cells,0.001989,0.036182,0.000410,0.000030,0.000889,6.116328e-04,0.000245,0.001251,...,1.159604e-04,5.572096e-04,0.003310,1.796931e-06,1.078609e-03,0.006590,0.009272,9.529314e-05,0.000183,0.000210
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg(diff),0.000398,0.034033,0.000040,0.000016,0.000528,8.011810e-05,0.000118,0.000312,...,2.621082e-04,4.515319e-04,0.004143,1.051052e-06,2.050176e-04,0.000995,0.032553,3.209589e-05,0.000204,0.000357
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,T cells,0.001481,0.000358,0.000126,0.000018,0.002222,3.951085e-06,0.000007,0.000223,...,3.578412e-04,7.166035e-05,0.000128,1.197694e-07,2.281222e-05,0.000166,0.000102,8.580065e-06,0.000145,0.000166
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,ILC3,0.012480,0.469688,0.000351,0.011805,0.007444,9.962440e-06,0.000606,0.001941,...,1.597248e-05,9.553372e-04,0.001987,6.983764e-07,2.646769e-05,0.000122,0.001447,1.538666e-03,0.002068,0.000369
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Tcm/Naive helper T cells,0.000743,0.148750,0.000104,0.000037,0.002329,2.954562e-04,0.000685,0.000771,...,8.786367e-05,3.405061e-04,0.008393,2.334269e-07,6.448736e-05,0.000524,0.001063,4.870436e-05,0.000104,0.000579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Naive B cells,0.000442,0.004714,0.000219,0.005828,0.001587,4.907626e-08,0.000111,0.000184,...,8.436597e-07,4.151164e-06,0.000168,3.167791e-06,3.019319e-07,0.000031,0.000039,9.836224e-07,0.001053,0.000004
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Transitional B cells,0.016038,0.082289,0.102189,0.001119,0.100390,6.797395e-08,0.000198,0.000169,...,2.736300e-04,1.484632e-06,0.000293,4.858899e-06,1.276056e-04,0.001648,0.074475,3.383468e-04,0.014351,0.008405
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,0.000005,0.000005,0.000009,0.004323,0.000184,4.129443e-09,0.000025,0.000002,...,4.365376e-06,1.888888e-07,0.000013,1.082049e-08,4.351355e-07,0.000025,0.000002,5.448122e-07,0.000011,0.000042
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,NK cells,0.002043,0.014295,0.000744,0.001508,0.002600,1.319137e-05,0.000016,0.000872,...,4.210057e-02,1.691473e-05,0.096445,8.662293e-04,1.989421e-03,0.003980,0.005356,1.587193e-03,0.001141,0.000534


In [22]:
scamp = pd.read_csv('scmap_all.csv', index_col=0)
scamp

,cellname,celltype,similarity
Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Myelocytes,0.568976
Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Immature B cells,0.771521
Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Early erythroid,0.883716
Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,HSC/MPP,0.875623
Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Germinal center B cells,0.825343
...,...,...,...
Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Tcm/Naive helper T cells,0.752742
Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,HSC/MPP,0.886624
Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Monocytes,0.532146
Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Mid erythroid,0.731983


In [44]:
print(set(actinn.index.tolist()) == set(rna.obs.index.tolist()))

True


# Majority voting

In [1]:
import pandas as pd
import scanpy as sc

## merge data

In [2]:
all = pd.read_csv('./new_data/combined_26.csv', index_col=0)
all

,chetah,scmap,SingleCellNet,scPred,singler
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP
...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP


In [3]:
all.columns = ['chetah', 'scmap', 'singlecellnet', 'scpred', 'singler']
all

,chetah,scmap,singlecellnet,scpred,singler
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP
...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP


In [4]:
celltypist = pd.read_csv('./new_data/celltypist.csv',index_col=0)
celltypist

,predicted_labels,over_clustering,majority_voting,conf_score
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Early erythroid,166,Early erythroid,0.347564
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Tcm/Naive helper T cells,181,Early erythroid,0.169708
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,CD16- NK cells,18,Early erythroid,0.257264
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Classical monocytes,14,Early erythroid,0.857415
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Regulatory T cells,181,Early erythroid,0.919876
...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,CD16- NK cells,27,CD16- NK cells,0.138753
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,CD16+ NK cells,53,Classical monocytes,0.000958
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Tcm/Naive helper T cells,123,Classical monocytes,0.004089
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Double-negative thymocytes,88,Classical monocytes,0.642792


In [5]:
all['celltypist'] = celltypist['predicted_labels'].tolist()
all

,chetah,scmap,singlecellnet,scpred,singler,celltypist
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP,Early erythroid
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP,Tcm/Naive helper T cells
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells,CD16- NK cells
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP,Classical monocytes
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP,Regulatory T cells
...,...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells,CD16- NK cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes,CD16+ NK cells
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP,Tcm/Naive helper T cells
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP,Double-negative thymocytes


In [6]:
actinn = pd.read_csv('./new_data/actinn.txt', sep='\t')
actinn

,cellname,celltype
0,Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Helper T cells
1,Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Helper T cells
2,Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Naive B cells
3,Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Helper T cells
4,Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,MAIT cells
...,...,...
53713,Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Memory B cells
53714,Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,ILC
53715,Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Helper T cells
53716,Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,MAIT cells


In [7]:
all['actinn'] = actinn['celltype'].tolist()
all

,chetah,scmap,singlecellnet,scpred,singler,celltypist,actinn
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP,Early erythroid,Helper T cells
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP,Tcm/Naive helper T cells,Helper T cells
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells,CD16- NK cells,Naive B cells
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP,Classical monocytes,Helper T cells
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP,Regulatory T cells,MAIT cells
...,...,...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells,CD16- NK cells,Memory B cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes,CD16+ NK cells,ILC
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP,Tcm/Naive helper T cells,Helper T cells
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP,Double-negative thymocytes,MAIT cells


In [8]:
scarch = pd.read_csv('./new_data/scarch.csv')
scarch

,Unnamed: 0,0
0,0,GMP
1,1,Cycling gamma-delta T cells
2,2,Late erythroid
3,3,Myelocytes
4,4,Myelocytes
...,...,...
53713,53713,NK cells
53714,53714,Neutrophils
53715,53715,Myelocytes
53716,53716,Myelocytes


In [9]:
all['scarch'] = scarch['0'].tolist()
all

,chetah,scmap,singlecellnet,scpred,singler,celltypist,actinn,scarch
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP,Early erythroid,Helper T cells,GMP
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP,Tcm/Naive helper T cells,Helper T cells,Cycling gamma-delta T cells
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells,CD16- NK cells,Naive B cells,Late erythroid
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP,Classical monocytes,Helper T cells,Myelocytes
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP,Regulatory T cells,MAIT cells,Myelocytes
...,...,...,...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells,CD16- NK cells,Memory B cells,NK cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes,CD16+ NK cells,ILC,Neutrophils
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP,Tcm/Naive helper T cells,Helper T cells,Myelocytes
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP,Double-negative thymocytes,MAIT cells,Myelocytes


In [10]:
all.to_csv('./method8_level3.csv')

## level_2 data

In [11]:
cell = pd.read_csv('./origin/sc_105.csv')
cell

,level_1,level_2,level_3
0,HSC/MPP,HSC/MPP,HSC/MPP
1,HSC/MPP,HSC/MPP,CMP
2,HSC/MPP,HSC/MPP,GMP
3,HSC/MPP,HSC/MPP,ELP
4,HSC/MPP,HSC/MPP,MEMP
...,...,...,...
100,Lymphoid,B cells,Pre-B cells
101,Lymphoid,T cells,T cells
102,Lymphoid,T cells,Cytotoxic T cells
103,Lymphoid,T cells,Tem/Effector cytotoxic T cells


In [12]:
result = dict(zip(cell['level_3'], cell['level_2']))
result

{'HSC/MPP': 'HSC/MPP',
 'CMP': 'HSC/MPP',
 'GMP': 'HSC/MPP',
 'ELP': 'HSC/MPP',
 'MEMP': 'HSC/MPP',
 'Megakaryocyte-erythroid-mast cell progenitor': 'HSC/MPP',
 'Neutrophil-myeloid progenitor': 'HSC/MPP',
 'Early lymphoid/T lymphoid': 'HSC/MPP',
 'CD8a/a': 'T cells',
 'CD8a/b(entry)': 'T cells',
 'Follicular helper T cells': 'T cells',
 'CRTAM+ gamma-delta T cells': 'T cells',
 'gamma-delta T cells': 'T cells',
 'MAIT cells': 'T cells',
 'Memory CD4+ cytotoxic T cells': 'T cells',
 'NKT cells': 'T cells',
 'Regulatory T cells': 'T cells',
 'T(agonist)': 'T cells',
 'Tcm/Naive cytotoxic T cells': 'T cells',
 'Tcm/Naive helper T cells': 'T cells',
 'Tem/Trm cytotoxic T cells': 'T cells',
 'Tem/Temra cytotoxic T cells': 'T cells',
 'Trm cytotoxic T cells': 'T cells',
 'Tem/Effector helper T cells': 'T cells',
 'Tem/Effector helper T cells PD1+': 'T cells',
 'Treg(diff)': 'T cells',
 'Type 1 helper T cells': 'T cells',
 'Type 17 helper T cells': 'T cells',
 'B cells': 'B cells',
 'Follicul

In [13]:
all_2 = all.replace(result)
all_2

,chetah,scmap,singlecellnet,scpred,singler,celltypist,actinn,scarch
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Granulocytes,Macrophages,Erythroid,HSC/MPP,Erythroid,T cells,HSC/MPP
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Granulocytes,T cells,HSC/MPP,HSC/MPP,T cells,T cells,Cycling cells
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Granulocytes,B cells,ETP,ILC,ILC,B cells,Erythroid
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Granulocytes,T cells,ILC,HSC/MPP,Monocytes,T cells,Myelocytes
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,B cells,Granulocytes,T cells,B cells,HSC/MPP,T cells,T cells,Myelocytes
...,...,...,...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Granulocytes,B cells,T cells,ILC,ILC,B cells,ILC
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Granulocytes,B cells,ILC,Promyelocytes,ILC,ILC,Granulocytes
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Granulocytes,T cells,Promyelocytes,HSC/MPP,T cells,T cells,Myelocytes
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Granulocytes,B cells,T cells,HSC/MPP,Double-negative thymocytes,T cells,Myelocytes


In [14]:
all_2.to_csv('./method8_level2.csv')

## preprocess for cell type

In [2]:
all = pd.read_csv('./scarch_all.csv', index_col=0)[['celltype']]
all.columns = ['scarch']
actinn = pd.read_csv('./actinn_all.csv', index_col=0)
all['actinn'] = actinn['celltype'].tolist()

FileNotFoundError: [Errno 2] No such file or directory: './scarch_all.csv'

In [ ]:
all

In [3]:
actinn.index.tolist() == all.index.tolist()

NameError: name 'actinn' is not defined

In [6]:
celltypist = pd.read_csv('../6_integrate/processed/celltypist/cellTypist.csv', index_col=0)
celltypist = celltypist.reindex(all.index.tolist())

In [7]:
celltypist.index.tolist() == all.index.tolist()

True

In [8]:
all['celltypist'] = celltypist['predicted_labels'].tolist()

In [9]:
all

,scarch,actinn,celltypist
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Regulatory T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Treg(diff),Tem/Effector helper T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,T cells,CD16- NK cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,ILC3,Tem/Trm cytotoxic T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Tcm/Naive helper T cells,Tcm/Naive helper T cells
...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Naive B cells,Small pre-B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Transitional B cells,Small pre-B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Memory B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,NK cells,Tem/Temra cytotoxic T cells


In [10]:
scmap = pd.read_csv('./scmap_all.csv', index_col=0)
scmap = scmap.reindex(all.index.tolist())
all['scmap'] = scmap['celltype'].tolist()

In [11]:
scmap.index.tolist() == all.index.tolist()

True

In [13]:
singcellnet = pd.read_csv('./singcellnet_all.csv', index_col=0)
singcellnet

,Cell,CellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Tcm/Naive helper T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Tcm/Naive helper T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Helper T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Tcm/Naive helper T cells
...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Naive B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,CD16+ NK cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Cytotoxic T cells


In [14]:
singcellnet.index.tolist() == all.index.tolist()

True

In [15]:
all['singcellnet'] = singcellnet['CellType'].tolist()

In [16]:
all

,scarch,actinn,celltypist,scmap,singcellnet
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Regulatory T cells,Regulatory T cells,unassigned,Tcm/Naive helper T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Treg(diff),Tem/Effector helper T cells,unassigned,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,T cells,CD16- NK cells,Neutrophils,Tcm/Naive helper T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,ILC3,Tem/Trm cytotoxic T cells,Neutrophils,Helper T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Tcm/Naive helper T cells,Tcm/Naive helper T cells,Type 17 helper T cells,Tcm/Naive helper T cells
...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Naive B cells,Small pre-B cells,Naive B cells,Naive B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Transitional B cells,Small pre-B cells,NK cells,CD16+ NK cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,NK cells,Tem/Temra cytotoxic T cells,NK cells,Cytotoxic T cells


In [19]:
scpred = pd.read_csv('./scpred_all.csv', index_col=0)
scpred = scpred.reindex(all.index.tolist())

In [20]:
scpred.index.tolist() == all.index.tolist()

True

In [21]:
scpred

,cellname,celltype
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Helper T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Helper T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Helper T cells
...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Naive B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Cytotoxic T cells


In [22]:
all['scpred'] = scpred['celltype'].tolist()

In [23]:
all

,scarch,actinn,celltypist,scmap,singcellnet,scpred
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Regulatory T cells,Regulatory T cells,unassigned,Tcm/Naive helper T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Treg(diff),Tem/Effector helper T cells,unassigned,Regulatory T cells,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,T cells,CD16- NK cells,Neutrophils,Tcm/Naive helper T cells,Helper T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,ILC3,Tem/Trm cytotoxic T cells,Neutrophils,Helper T cells,Helper T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Tcm/Naive helper T cells,Tcm/Naive helper T cells,Type 17 helper T cells,Tcm/Naive helper T cells,Helper T cells
...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Naive B cells,Small pre-B cells,Naive B cells,Naive B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Transitional B cells,Small pre-B cells,NK cells,CD16+ NK cells,Naive B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,NK cells,Tem/Temra cytotoxic T cells,NK cells,Cytotoxic T cells,Cytotoxic T cells


In [24]:
all.to_csv('./all_method6.csv')

In [17]:
all = pd.read_csv('./all_method6.csv', index_col=0)
all

,scarch,actinn,celltypist,scmap,singcellnet,scpred,mv_3
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Regulatory T cells,Regulatory T cells,unassigned,Tcm/Naive helper T cells,Regulatory T cells,notAvailable
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Treg(diff),Tem/Effector helper T cells,unassigned,Regulatory T cells,Regulatory T cells,notAvailable
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,T cells,CD16- NK cells,Neutrophils,Tcm/Naive helper T cells,Helper T cells,notAvailable
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,ILC3,Tem/Trm cytotoxic T cells,Neutrophils,Helper T cells,Helper T cells,notAvailable
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Tcm/Naive helper T cells,Tcm/Naive helper T cells,Type 17 helper T cells,Tcm/Naive helper T cells,Helper T cells,notAvailable
...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Naive B cells,Small pre-B cells,Naive B cells,Naive B cells,Memory B cells,notAvailable
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Transitional B cells,Small pre-B cells,NK cells,CD16+ NK cells,Naive B cells,notAvailable
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,NK cells,Tem/Temra cytotoxic T cells,NK cells,Cytotoxic T cells,Cytotoxic T cells,notAvailable


In [18]:
sr = pd.read_csv('/pool1/liumengmeng/project_blood_scATAC/cell_annotation/combined_1-25/sr_combined_data-1-25.csv', index_col=2)
sr

,Unnamed: 0,X,V2
V1,,,
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,predicted_celltype_1.csv.1,1,Tem/Effector helper T cells PD1+
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,predicted_celltype_1.csv.2,2,ILC2
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,predicted_celltype_1.csv.3,3,ILC2
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,predicted_celltype_1.csv.4,4,Megakaryocyte precursor
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,predicted_celltype_1.csv.5,5,Tem/Effector helper T cells PD1+
...,...,...,...
Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,predicted_celltype_9.csv.99996,99996,Tcm/Naive helper T cells
Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,predicted_celltype_9.csv.99997,99997,HSC/MPP
Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,predicted_celltype_9.csv.99998,99998,Non-classical monocytes


In [19]:
sr.index.tolist() == all.index.tolist()

False

In [20]:
sr = sr.reindex(all.index.tolist())

In [21]:
sr.index.tolist() == all.index.tolist()

True

In [22]:
all['singler'] = sr['V2'].tolist()
all

,scarch,actinn,celltypist,scmap,singcellnet,scpred,mv_3,singler
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Regulatory T cells,Regulatory T cells,unassigned,Tcm/Naive helper T cells,Regulatory T cells,notAvailable,Tem/Effector helper T cells PD1+
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Treg(diff),Tem/Effector helper T cells,unassigned,Regulatory T cells,Regulatory T cells,notAvailable,ILC2
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,T cells,CD16- NK cells,Neutrophils,Tcm/Naive helper T cells,Helper T cells,notAvailable,ILC2
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,ILC3,Tem/Trm cytotoxic T cells,Neutrophils,Helper T cells,Helper T cells,notAvailable,Megakaryocyte precursor
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Tcm/Naive helper T cells,Tcm/Naive helper T cells,Type 17 helper T cells,Tcm/Naive helper T cells,Helper T cells,notAvailable,Tem/Effector helper T cells PD1+
...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Naive B cells,Small pre-B cells,Naive B cells,Naive B cells,Memory B cells,notAvailable,Naive B cells
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Transitional B cells,Small pre-B cells,NK cells,CD16+ NK cells,Naive B cells,notAvailable,Immature B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,NK cells,Tem/Temra cytotoxic T cells,NK cells,Cytotoxic T cells,Cytotoxic T cells,notAvailable,Cytotoxic T cells


In [23]:
chetah = pd.read_csv('./huizong/chetah_all.csv',index_col=0)
chetah

,cellname,celltype
Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Weng_Nature_2024_4-AGCATCCCAGCACCAT-1,Unassigned
Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Weng_Nature_2024_4-AGCATCCCAGGACCAA-2,Immature B cells
Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Weng_Nature_2024_4-AGCATCCCATATTGAC-3,Mid erythroid
Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Weng_Nature_2024_4-AGCATCCCATCGCTTT-3,Mid erythroid
Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Weng_Nature_2024_4-AGCATCCCATGAAATG-2,Mid erythroid
...,...,...
Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Weng_Nature_2024_4-AGCATCCCAAATGCCC-1,Megakaryocytes/platelets
Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Weng_Nature_2024_4-AGCATCCCAACAGGTG-3,Node10
Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Weng_Nature_2024_4-AGCATCCCAATCGCAC-1,Unassigned
Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Weng_Nature_2024_4-AGCATCCCACCTACGG-2,Immature B cells


In [24]:
chetah.index.tolist() == all.index.tolist()

False

In [25]:
chetah = chetah.reindex(all.index.tolist())

In [26]:
chetah.index.tolist() == all.index.tolist()

True

In [27]:
all['chetah'] = chetah['celltype'].tolist()
all

,scarch,actinn,celltypist,scmap,singcellnet,scpred,mv_3,singler,chetah
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Regulatory T cells,Regulatory T cells,unassigned,Tcm/Naive helper T cells,Regulatory T cells,notAvailable,Tem/Effector helper T cells PD1+,Treg(diff)
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Treg(diff),Tem/Effector helper T cells,unassigned,Regulatory T cells,Regulatory T cells,notAvailable,ILC2,Early lymphoid/T lymphoid
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,T cells,CD16- NK cells,Neutrophils,Tcm/Naive helper T cells,Helper T cells,notAvailable,ILC2,Treg(diff)
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,ILC3,Tem/Trm cytotoxic T cells,Neutrophils,Helper T cells,Helper T cells,notAvailable,Megakaryocyte precursor,Treg(diff)
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Tcm/Naive helper T cells,Tcm/Naive helper T cells,Type 17 helper T cells,Tcm/Naive helper T cells,Helper T cells,notAvailable,Tem/Effector helper T cells PD1+,Treg(diff)
...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Naive B cells,Small pre-B cells,Naive B cells,Naive B cells,Memory B cells,notAvailable,Naive B cells,Unassigned
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Transitional B cells,Small pre-B cells,NK cells,CD16+ NK cells,Naive B cells,notAvailable,Immature B cells,Immature B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,NK cells,Tem/Temra cytotoxic T cells,NK cells,Cytotoxic T cells,Cytotoxic T cells,notAvailable,Cytotoxic T cells,Megakaryocytes/platelets


In [28]:
del all['mv_3']

In [30]:
all.to_csv('./all_method8.csv')

## preprocess for cell type level2

In [16]:
all = pd.read_csv('./all_method8.csv', index_col=0)
all

,scarch,actinn,celltypist,scmap,singcellnet,scpred,singler,chetah
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Neutrophils,Regulatory T cells,Regulatory T cells,unassigned,Tcm/Naive helper T cells,Regulatory T cells,Tem/Effector helper T cells PD1+,Treg(diff)
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Neutrophils,Treg(diff),Tem/Effector helper T cells,unassigned,Regulatory T cells,Regulatory T cells,ILC2,Early lymphoid/T lymphoid
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Neutrophils,T cells,CD16- NK cells,Neutrophils,Tcm/Naive helper T cells,Helper T cells,ILC2,Treg(diff)
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Neutrophils,ILC3,Tem/Trm cytotoxic T cells,Neutrophils,Helper T cells,Helper T cells,Megakaryocyte precursor,Treg(diff)
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Neutrophils,Tcm/Naive helper T cells,Tcm/Naive helper T cells,Type 17 helper T cells,Tcm/Naive helper T cells,Helper T cells,Tem/Effector helper T cells PD1+,Treg(diff)
...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,Follicular B cells,Naive B cells,Small pre-B cells,Naive B cells,Naive B cells,Memory B cells,Naive B cells,Unassigned
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,Transitional B cells,Small pre-B cells,NK cells,CD16+ NK cells,Naive B cells,Immature B cells,Immature B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,Memory B cells,B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,NK cells,Tem/Temra cytotoxic T cells,NK cells,Cytotoxic T cells,Cytotoxic T cells,Cytotoxic T cells,Megakaryocytes/platelets


In [17]:
cell = pd.read_csv('./origin/sc_105.csv')
cell

,level_1,level_2,level_3
0,HSC/MPP,HSC/MPP,HSC/MPP
1,HSC/MPP,HSC/MPP,CMP
2,HSC/MPP,HSC/MPP,GMP
3,HSC/MPP,HSC/MPP,ELP
4,HSC/MPP,HSC/MPP,MEMP
...,...,...,...
100,Lymphoid,B cells,Pre-B cells
101,Lymphoid,T cells,T cells
102,Lymphoid,T cells,Cytotoxic T cells
103,Lymphoid,T cells,Tem/Effector cytotoxic T cells


In [18]:
result = dict(zip(cell['level_3'], cell['level_2']))
result

{'HSC/MPP': 'HSC/MPP',
 'CMP': 'HSC/MPP',
 'GMP': 'HSC/MPP',
 'ELP': 'HSC/MPP',
 'MEMP': 'HSC/MPP',
 'Megakaryocyte-erythroid-mast cell progenitor': 'HSC/MPP',
 'Neutrophil-myeloid progenitor': 'HSC/MPP',
 'Early lymphoid/T lymphoid': 'HSC/MPP',
 'CD8a/a': 'T cells',
 'CD8a/b(entry)': 'T cells',
 'Follicular helper T cells': 'T cells',
 'CRTAM+ gamma-delta T cells': 'T cells',
 'gamma-delta T cells': 'T cells',
 'MAIT cells': 'T cells',
 'Memory CD4+ cytotoxic T cells': 'T cells',
 'NKT cells': 'T cells',
 'Regulatory T cells': 'T cells',
 'T(agonist)': 'T cells',
 'Tcm/Naive cytotoxic T cells': 'T cells',
 'Tcm/Naive helper T cells': 'T cells',
 'Tem/Trm cytotoxic T cells': 'T cells',
 'Tem/Temra cytotoxic T cells': 'T cells',
 'Trm cytotoxic T cells': 'T cells',
 'Tem/Effector helper T cells': 'T cells',
 'Tem/Effector helper T cells PD1+': 'T cells',
 'Treg(diff)': 'T cells',
 'Type 1 helper T cells': 'T cells',
 'Type 17 helper T cells': 'T cells',
 'B cells': 'B cells',
 'Follicul

In [19]:
all_2 = all.replace(result)
all_2

,scarch,actinn,celltypist,scmap,singcellnet,scpred,singler,chetah
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Granulocytes,T cells,T cells,unassigned,T cells,T cells,T cells,T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Granulocytes,T cells,T cells,unassigned,T cells,T cells,ILC,HSC/MPP
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Granulocytes,T cells,ILC,Granulocytes,T cells,T cells,ILC,T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Granulocytes,ILC,T cells,Granulocytes,T cells,T cells,Megakaryocyte precursor,T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Granulocytes,T cells,T cells,T cells,T cells,T cells,T cells,T cells
...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,B cells,B cells,B-cell lineage,B cells,B cells,B cells,B cells,Unassigned
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,Myelocytes,B cells,B-cell lineage,ILC,ILC,B cells,B cells,B cells
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,B cells,B cells,B cells,B cells,B cells,B cells,B cells,B cells
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Myelocytes,ILC,T cells,ILC,T cells,T cells,T cells,Megakaryocytes/platelets


In [20]:
all_2.to_csv('all_method8_level2.csv')

## compute

In [ ]:
import pandas as pd

# Vectorized implementation of majority voting
def majority_vote_vectorized(df):
    # Count the frequency of each value per row
    mode_counts = df.apply(lambda row: row.value_counts(), axis=1).fillna(0)

    # Find the most frequent value and its occurrence count
    max_counts = mode_counts.max(axis=1)
    majority_values = mode_counts.idxmax(axis=1)

    # Determine validity based on the occurrence count (threshold >= 4)
    result = majority_values.where(max_counts >= 4, "notAvailable")
    return result

all = pd.read_csv('./all_method6.csv', index_col=0)
all['mv_3'] = majority_vote_vectorized(all)
all.to_csv('./all_method6.csv')

all = pd.read_csv('./all_method6_level2.csv', index_col=0)
all['mv_3'] = majority_vote_vectorized(all)
all.to_csv('./all_method6_level2.csv')

# Process for the new datasets

In [1]:
import pandas as pd
import scanpy as sc

## merge

In [2]:
all = pd.read_csv('./new_data/combined_26.csv', index_col=0)
all

,chetah,scmap,SingleCellNet,scPred,singler
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP
...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP


In [3]:
all.columns = ['chetah', 'scmap', 'singlecellnet', 'scpred', 'singler']
all

,chetah,scmap,singlecellnet,scpred,singler
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP
...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP


In [4]:
celltypist = pd.read_csv('./new_data/celltypist.csv',index_col=0)
celltypist

,predicted_labels,over_clustering,majority_voting,conf_score
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Early erythroid,166,Early erythroid,0.347564
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Tcm/Naive helper T cells,181,Early erythroid,0.169708
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,CD16- NK cells,18,Early erythroid,0.257264
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Classical monocytes,14,Early erythroid,0.857415
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Regulatory T cells,181,Early erythroid,0.919876
...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,CD16- NK cells,27,CD16- NK cells,0.138753
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,CD16+ NK cells,53,Classical monocytes,0.000958
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Tcm/Naive helper T cells,123,Classical monocytes,0.004089
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Double-negative thymocytes,88,Classical monocytes,0.642792


In [5]:
all['celltypist'] = celltypist['predicted_labels'].tolist()
all

,chetah,scmap,singlecellnet,scpred,singler,celltypist
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP,Early erythroid
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP,Tcm/Naive helper T cells
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells,CD16- NK cells
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP,Classical monocytes
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP,Regulatory T cells
...,...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells,CD16- NK cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes,CD16+ NK cells
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP,Tcm/Naive helper T cells
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP,Double-negative thymocytes


In [6]:
actinn = pd.read_csv('./new_data/actinn.txt', sep='\t')
actinn

,cellname,celltype
0,Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Helper T cells
1,Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Helper T cells
2,Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Naive B cells
3,Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Helper T cells
4,Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,MAIT cells
...,...,...
53713,Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Memory B cells
53714,Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,ILC
53715,Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Helper T cells
53716,Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,MAIT cells


In [7]:
all['actinn'] = actinn['celltype'].tolist()
all

,chetah,scmap,singlecellnet,scpred,singler,celltypist,actinn
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP,Early erythroid,Helper T cells
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP,Tcm/Naive helper T cells,Helper T cells
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells,CD16- NK cells,Naive B cells
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP,Classical monocytes,Helper T cells
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP,Regulatory T cells,MAIT cells
...,...,...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells,CD16- NK cells,Memory B cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes,CD16+ NK cells,ILC
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP,Tcm/Naive helper T cells,Helper T cells
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP,Double-negative thymocytes,MAIT cells


In [8]:
scarch = pd.read_csv('./new_data/scarch.csv')
scarch

,Unnamed: 0,0
0,0,GMP
1,1,Cycling gamma-delta T cells
2,2,Late erythroid
3,3,Myelocytes
4,4,Myelocytes
...,...,...
53713,53713,NK cells
53714,53714,Neutrophils
53715,53715,Myelocytes
53716,53716,Myelocytes


In [9]:
all['scarch'] = scarch['0'].tolist()
all

,chetah,scmap,singlecellnet,scpred,singler,celltypist,actinn,scarch
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Neutrophils,Macrophages,Early erythroid,MEMP,Early erythroid,Helper T cells,GMP
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Neutrophils,Tcm/Naive helper T cells,MEMP,MEMP,Tcm/Naive helper T cells,Helper T cells,Cycling gamma-delta T cells
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Neutrophils,Memory B cells,ETP,CD16- NK cells,CD16- NK cells,Naive B cells,Late erythroid
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Neutrophils,Helper T cells,NK cells,HSC/MPP,Classical monocytes,Helper T cells,Myelocytes
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,Immature B cells,Neutrophils,Tcm/Naive helper T cells,Immature B cells,MEMP,Regulatory T cells,MAIT cells,Myelocytes
...,...,...,...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Neutrophils,Memory B cells,Helper T cells,CD16- NK cells,CD16- NK cells,Memory B cells,NK cells
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Neutrophils,Memory B cells,ILC,Promyelocytes,CD16+ NK cells,ILC,Neutrophils
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Neutrophils,Helper T cells,Promyelocytes,GMP,Tcm/Naive helper T cells,Helper T cells,Myelocytes
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Neutrophils,Memory B cells,Tcm/Naive helper T cells,MEMP,Double-negative thymocytes,MAIT cells,Myelocytes


In [10]:
all.to_csv('./method8_level3.csv')

## merge level_2 data

In [11]:
cell = pd.read_csv('./origin/sc_105.csv')
cell

,level_1,level_2,level_3
0,HSC/MPP,HSC/MPP,HSC/MPP
1,HSC/MPP,HSC/MPP,CMP
2,HSC/MPP,HSC/MPP,GMP
3,HSC/MPP,HSC/MPP,ELP
4,HSC/MPP,HSC/MPP,MEMP
...,...,...,...
100,Lymphoid,B cells,Pre-B cells
101,Lymphoid,T cells,T cells
102,Lymphoid,T cells,Cytotoxic T cells
103,Lymphoid,T cells,Tem/Effector cytotoxic T cells


In [12]:
result = dict(zip(cell['level_3'], cell['level_2']))
result

{'HSC/MPP': 'HSC/MPP',
 'CMP': 'HSC/MPP',
 'GMP': 'HSC/MPP',
 'ELP': 'HSC/MPP',
 'MEMP': 'HSC/MPP',
 'Megakaryocyte-erythroid-mast cell progenitor': 'HSC/MPP',
 'Neutrophil-myeloid progenitor': 'HSC/MPP',
 'Early lymphoid/T lymphoid': 'HSC/MPP',
 'CD8a/a': 'T cells',
 'CD8a/b(entry)': 'T cells',
 'Follicular helper T cells': 'T cells',
 'CRTAM+ gamma-delta T cells': 'T cells',
 'gamma-delta T cells': 'T cells',
 'MAIT cells': 'T cells',
 'Memory CD4+ cytotoxic T cells': 'T cells',
 'NKT cells': 'T cells',
 'Regulatory T cells': 'T cells',
 'T(agonist)': 'T cells',
 'Tcm/Naive cytotoxic T cells': 'T cells',
 'Tcm/Naive helper T cells': 'T cells',
 'Tem/Trm cytotoxic T cells': 'T cells',
 'Tem/Temra cytotoxic T cells': 'T cells',
 'Trm cytotoxic T cells': 'T cells',
 'Tem/Effector helper T cells': 'T cells',
 'Tem/Effector helper T cells PD1+': 'T cells',
 'Treg(diff)': 'T cells',
 'Type 1 helper T cells': 'T cells',
 'Type 17 helper T cells': 'T cells',
 'B cells': 'B cells',
 'Follicul

In [13]:
all_2 = all.replace(result)
all_2

,chetah,scmap,singlecellnet,scpred,singler,celltypist,actinn,scarch
Marderstein_Nature_2024-AAACAGCCAGTTATCG-TS21_4_H,Unassigned,Granulocytes,Macrophages,Erythroid,HSC/MPP,Erythroid,T cells,HSC/MPP
Marderstein_Nature_2024-AAACCGAAGTCCTGAG-TS21_4_H,Unassigned,Granulocytes,T cells,HSC/MPP,HSC/MPP,T cells,T cells,Cycling cells
Marderstein_Nature_2024-AAACGCGCAGCTAACC-TS21_4_H,Unassigned,Granulocytes,B cells,ETP,ILC,ILC,B cells,Erythroid
Marderstein_Nature_2024-AAACGGATCATTACTC-TS21_4_H,Unassigned,Granulocytes,T cells,ILC,HSC/MPP,Monocytes,T cells,Myelocytes
Marderstein_Nature_2024-AAACGGATCCTCCTAA-TS21_4_H,B cells,Granulocytes,T cells,B cells,HSC/MPP,T cells,T cells,Myelocytes
...,...,...,...,...,...,...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Node12,Granulocytes,B cells,T cells,ILC,ILC,B cells,ILC
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Unassigned,Granulocytes,B cells,ILC,Promyelocytes,ILC,ILC,Granulocytes
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Unassigned,Granulocytes,T cells,Promyelocytes,HSC/MPP,T cells,T cells,Myelocytes
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Unassigned,Granulocytes,B cells,T cells,HSC/MPP,Double-negative thymocytes,T cells,Myelocytes


In [14]:
all_2.to_csv('./method8_level2.csv')